# Forex Data Analysis

This notebook reads EUR/USD exchange rate data and displays it in a table format.

In [80]:
import pandas as pd
import numpy as np

# Read the EUR/USD data from CSV file
df = pd.read_csv('data/eurusd.csv')

# Display basic information about the data
print(f"Trades: {df.shape[0]}")
print(f"Columns: {df.columns.tolist()}")

Trades: 933
Columns: ['Date', 'Trade', 'Direction', 'EMA', 'SL', 'Pullback', 'TP', 'BOS/CH']


In [81]:
# Display the data table for debugging reasons
df

,Date,Trade,Direction,EMA,SL,Pullback,TP,BOS/CH
0,2025-02-03,#1,Sell,Sell,10.7,10.7,0,BOS
1,2025-02-03,#2,Buy,Buy,5.4,5.4,0,BOS
2,2025-02-03,#4,Buy,Buy,19.3,1.7,24,BOS
3,2025-02-04,#1,Buy,Buy,6.7,0.5,70,BOS
4,2025-02-04,#2,Buy,Sell,7.6,7.6,0,BOS
...,...,...,...,...,...,...,...,...
928,2025-08-08,#2,Sell,Sell,4.0,4.0,0,BOS
929,2025-08-08,#3,Buy,Sell,6.3,6.3,0,CH
930,2025-08-08,#4,Sell,Buy,1.8,1.8,0,CH
931,2025-08-08,#5,Sell,Buy,5.1,5.1,0,CH


In [82]:
# Define RRR calculation function
def calculate_rrr_stats(data_df, strategy_name):
    """
    Calculate Risk-Reward Ratio statistics for a given dataframe and strategy.
    
    Args:
        data_df: DataFrame containing trading data
        strategy_name: Name of the strategy for labeling
    
    Returns:
        DataFrame with RRR statistics
    """
    total_trades = len(data_df)
    
    if total_trades == 0:
        return pd.DataFrame({
            strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome'],
            '1:1 RRR': [0, 0, 0, '0.0%', '0.0%', '0R']
        })
    
    rrr_configs = [
        (1, 50.0),   # 1:1 RRR needs 50% win rate to break even
        (2, 33.0),   # 1:2 RRR needs 33% win rate to break even
        (3, 25.0),   # 1:3 RRR needs 25% win rate to break even
        (4, 20.0),   # 1:4 RRR needs 20% win rate to break even
        (5, 17.0),   # 1:5 RRR needs 17% win rate to break even
        (10, 9.0),   # 1:10 RRR needs 9% win rate to break even
    ]
    
    summary_data = {
        strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
    }
    
    for ratio, breakeven_rate in rrr_configs:
        profitable = data_df[data_df['TP'] > ratio * data_df['SL']]
        wins = len(profitable)
        losses = total_trades - wins
        win_rate = (wins / total_trades) * 100
        edge = win_rate - breakeven_rate
        outcome = (wins * ratio) - losses
        
        summary_data[f'1:{ratio} RRR'] = [
            total_trades,
            wins,
            losses,
            f"{win_rate:.1f}%",
            f"{edge:.1f}%",
            f"{outcome}R"
        ]
    
    return pd.DataFrame(summary_data)

# Define strategy configurations
class Strategy:
    def __init__(self, name, filter_func, description=""):
        self.name = name
        self.filter_func = filter_func
        self.description = description
    
    def apply(self, df):
        """Apply the strategy filter to the dataframe."""
        return self.filter_func(df)

## Strategy Backtesting System

This code provides a modular system for backtesting forex strategies:

### Key Features:
1. **Reusable RRR Calculator**: A single function calculates all Risk-Reward Ratios (1:1 to 1:10)
2. **Strategy Class**: Simple strategy definition with name, filter function, and description
3. **Easy Extension**: Add new strategies by simply appending to the strategies list
4. **No Code Duplication**: Each strategy uses the same calculation logic
5. **Dynamic Display**: All strategies are automatically calculated and displayed

### How to Add New Strategies:
Simply add a new Strategy object with:
- **Name**: Display name for the strategy
- **Filter Function**: Lambda function that filters the dataframe
- **Description**: Optional description of the strategy logic

Example:
```python
strategies.append(
    Strategy(
        "My New Strategy",
        lambda df: df[df['SL'] < 10],  # Your filter logic here
        "Description of what this strategy does"
    )
)
```

In [83]:
# Define all strategies using the new system
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "All trades without any filtering"
    ),
    Strategy(
        "1pip Pullback Strategy",
        lambda df: df[df['Pullback'] >= 1],
        "Trades with at least 1 pip pullback"
    ),
    Strategy(
        "BOS Strategy",
        lambda df: df[df['BOS/CH'] == 'BOS'],
        "Trades with Break of Structure"
    ),
    Strategy(
        "CH Strategy",
        lambda df: df[df['BOS/CH'] == 'CH'],
        "Trades with Change Of Character"
    ),
    Strategy(
        "EMA Strategy",
        lambda df: df[df['EMA'] == df['Direction']],
        "Trades where EMA aligns with trade direction"
    ),
    Strategy(
        "EMA + BOS Strategy",
        lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')],
        "Trades with EMA and BOS alignment"
    ),
    Strategy(
        "EMA + CH Strategy",
        lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH')],
        "Trades with EMA and CH alignment"
    ),
    Strategy(
        "Opposite EMA + CH Strategy",
        lambda df: df[(df['EMA'] != df['Direction']) & (df['BOS/CH'] == 'CH')],
        "Trades with opposite EMA and CH alignment"
    ),
]

In [84]:
# Advanced: Pullback strategies
def create_pullback_strategies(pullback_values):
    """Generate multiple pullback strategies with different thresholds."""
    pullback_strategies = []
    for value in pullback_values:
        pullback_strategies.append(
            Strategy(
                f"{value}pip Pullback",
                lambda df, v=value: df[df['Pullback'] >= v],
                f"Trades with at least {value} pip pullback"
            )
        )
    return pullback_strategies

# Advanced: Up To SL strategies
def create_limit_sl_strategies(limit_sl_values):
    """Generate multiple limit SL strategies with different thresholds."""
    limit_sl_strategies = []
    for value in limit_sl_values:
        limit_sl_strategies.append(
            Strategy(
                f"Up to {value} pip SL",
                lambda df, v=value: df[df['SL'] <= v],
                f"Trades with at most {value} pip SL"
            )
        )
    return limit_sl_strategies

# Advanced: SL bigger than pips strategies
def create_sl_bigger_than_pips_strategies(sl_bigger_than_pips_values):
    """Generate multiple SL bigger than pips strategies with different thresholds."""
    sl_bigger_than_pips_strategies = []
    for value in sl_bigger_than_pips_values:
        sl_bigger_than_pips_strategies.append(
            Strategy(
                f"SL bigger than {value} pips",
                lambda df, v=value: df[df['SL'] > v],
                f"Trades with SL bigger than {value} pips"
            )
        )
    return sl_bigger_than_pips_strategies

# Uncomment to add multiple pullback strategies at once:
strategies.extend(create_pullback_strategies([1, 2, 3, 4, 5, 10, 15]))
strategies.extend(create_limit_sl_strategies([2, 3, 4, 5, 10, 15]))
strategies.extend(create_sl_bigger_than_pips_strategies([2, 3, 4, 5, 10, 15]))

# Calculate all strategy results
strategy_results = {}
for strategy in strategies:
    filtered_df = strategy.apply(df)
    summary_df = calculate_rrr_stats(filtered_df, strategy.name)
    strategy_results[strategy.name] = summary_df

# Function to get top strategies for a specific RRR
def get_top_strategies_for_rrr(strategy_results, rrr_column, top_n=5):
    """Get top N strategies for a specific RRR based on outcome."""
    strategy_performance = []
    
    for strategy_name, df in strategy_results.items():
        # Extract metrics
        total_trades = df[rrr_column].iloc[0]
        wins = df[rrr_column].iloc[1]
        losses = df[rrr_column].iloc[2]
        win_rate = df[rrr_column].iloc[3]
        edge = df[rrr_column].iloc[4]
        outcome_str = df[rrr_column].iloc[5]
        outcome = int(outcome_str.replace('R', ''))
        
        strategy_performance.append({
            'Strategy': strategy_name,
            'Trades': total_trades,
            'Wins': wins,
            'Losses': losses,
            'Win Rate': win_rate,
            'Edge': edge,
            'Outcome': outcome_str,
            'outcome_value': outcome  # For sorting
        })
    
    # Sort by outcome and get top N
    top_strategies = sorted(strategy_performance, key=lambda x: x['outcome_value'], reverse=True)[:top_n]
    
    # Remove the sorting key from display
    for strat in top_strategies:
        del strat['outcome_value']
    
    return pd.DataFrame(top_strategies)

# Display top 5 strategies for each RRR in separate tables
print("## Best performing strategies by RRR:\n")

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
    ('1:4 RRR', '1:4'),
    ('1:5 RRR', '1:5'),
    ('1:10 RRR', '1:10')
]

for rrr_column, rrr_label in rrr_configs:
    print(f"### Top 5 Strategies for {rrr_label} Risk-Reward Ratio")
    top_5_df = get_top_strategies_for_rrr(strategy_results, rrr_column, top_n=5)
    display(top_5_df)
    print()  # Add spacing between tables

## Best performing strategies by RRR:

### Top 5 Strategies for 1:1 Risk-Reward Ratio


,Strategy,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,SL bigger than 5 pips,491,257,234,52.3%,2.3%,23R
1,SL bigger than 10 pips,110,56,54,50.9%,0.9%,2R
2,SL bigger than 15 pips,36,19,17,52.8%,2.8%,2R
3,SL bigger than 4 pips,629,312,317,49.6%,-0.4%,-5R
4,EMA + CH Strategy,92,42,50,45.7%,-4.3%,-8R



### Top 5 Strategies for 1:2 Risk-Reward Ratio


,Strategy,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,SL bigger than 5 pips,491,184,307,37.5%,4.5%,61R
1,SL bigger than 4 pips,629,222,407,35.3%,2.3%,37R
2,SL bigger than 3 pips,790,274,516,34.7%,1.7%,32R
3,EMA Strategy,600,207,393,34.5%,1.5%,21R
4,EMA + BOS Strategy,508,176,332,34.6%,1.6%,20R



### Top 5 Strategies for 1:3 Risk-Reward Ratio


...

,Strategy,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,SL bigger than 5 pips,491,102,389,20.8%,0.8%,19R
1,EMA + BOS Strategy,508,102,406,20.1%,0.1%,2R
2,Up to 2 pip SL,50,10,40,20.0%,0.0%,0R
3,SL bigger than 15 pips,36,7,29,19.4%,-0.6%,-1R
4,Up to 3 pip SL,143,28,115,19.6%,-0.4%,-3R



### Top 5 Strategies for 1:5 Risk-Reward Ratio


,Strategy,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Up to 4 pip SL,304,54,250,17.8%,0.8%,20R
1,Up to 3 pip SL,143,27,116,18.9%,1.9%,19R
2,Up to 2 pip SL,50,10,40,20.0%,3.0%,10R
3,EMA + CH Strategy,92,16,76,17.4%,0.4%,4R
4,CH Strategy,282,45,237,16.0%,-1.0%,-12R



### Top 5 Strategies for 1:10 Risk-Reward Ratio


,Strategy,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,Up to 2 pip SL,50,4,46,8.0%,-1.0%,-6R
1,15pip Pullback,17,0,17,0.0%,-9.0%,-17R
2,SL bigger than 15 pips,36,0,36,0.0%,-9.0%,-36R
3,Up to 3 pip SL,143,9,134,6.3%,-2.7%,-44R
4,10pip Pullback,63,0,63,0.0%,-9.0%,-63R


In [85]:
# Display all strategy results
for strategy_name, summary_df in strategy_results.items():
    # Style the first column to have a fixed width
    styled_df = summary_df.style.set_properties(
        subset=[strategy_name], 
        **{'width': '220px'}
    )
    display(styled_df)
    print('')  # Add spacing between tables

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,933,933,933,933,933,933
1,Wins,407,310,247,182,147,30
2,Losses,526,623,686,751,786,903
3,Win Rate,43.6%,33.2%,26.5%,19.5%,15.8%,3.2%
4,Edge,-6.4%,0.2%,1.5%,-0.5%,-1.2%,-5.8%
5,Outcome,-119R,-3R,55R,-23R,-51R,-603R


,1pip Pullback Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,824,824,824,824,824,824
1,Wins,307,230,182,136,108,19
2,Losses,517,594,642,688,716,805
3,Win Rate,37.3%,27.9%,22.1%,16.5%,13.1%,2.3%
4,Edge,-12.7%,-5.1%,-2.9%,-3.5%,-3.9%,-6.7%
5,Outcome,-210R,-134R,-96R,-144R,-176R,-615R


,BOS Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,651,651,651,651,651,651
1,Wins,279,217,172,129,102,23
2,Losses,372,434,479,522,549,628
3,Win Rate,42.9%,33.3%,26.4%,19.8%,15.7%,3.5%
4,Edge,-7.1%,0.3%,1.4%,-0.2%,-1.3%,-5.5%
5,Outcome,-93R,0R,37R,-6R,-39R,-398R


,CH Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,282,282,282,282,282,282
1,Wins,128,93,75,53,45,7
2,Losses,154,189,207,229,237,275
3,Win Rate,45.4%,33.0%,26.6%,18.8%,16.0%,2.5%
4,Edge,-4.6%,-0.0%,1.6%,-1.2%,-1.0%,-6.5%
5,Outcome,-26R,-3R,18R,-17R,-12R,-205R


,EMA Strategy,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,600,600,600,600,600,600
1,Wins,270,207,165,119,96,21
2,Losses,330,393,435,481,504,579
3,Win Rate,45.0%,34.5%,27.5%,19.8%,16.0%,3.5%
4,Edge,-5.0%,1.5%,2.5%,-0.2%,-1.0%,-5.5%
5,Outcome,-60R,21R,60R,-5R,-24R,-369R


...

,SL bigger than 3 pips,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,790,790,790,790,790,790
1,Wins,371,274,217,154,120,21
2,Losses,419,516,573,636,670,769
3,Win Rate,47.0%,34.7%,27.5%,19.5%,15.2%,2.7%
4,Edge,-3.0%,1.7%,2.5%,-0.5%,-1.8%,-6.3%
5,Outcome,-48R,32R,78R,-20R,-70R,-559R


,SL bigger than 4 pips,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,629,629,629,629,629,629
1,Wins,312,222,173,123,93,11
2,Losses,317,407,456,506,536,618
3,Win Rate,49.6%,35.3%,27.5%,19.6%,14.8%,1.7%
4,Edge,-0.4%,2.3%,2.5%,-0.4%,-2.2%,-7.3%
5,Outcome,-5R,37R,63R,-14R,-71R,-508R


,SL bigger than 5 pips,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,491,491,491,491,491,491
1,Wins,257,184,143,102,77,6
2,Losses,234,307,348,389,414,485
3,Win Rate,52.3%,37.5%,29.1%,20.8%,15.7%,1.2%
4,Edge,2.3%,4.5%,4.1%,0.8%,-1.3%,-7.8%
5,Outcome,23R,61R,81R,19R,-29R,-425R


,SL bigger than 10 pips,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,110,110,110,110,110,110
1,Wins,56,34,27,18,13,0
2,Losses,54,76,83,92,97,110
3,Win Rate,50.9%,30.9%,24.5%,16.4%,11.8%,0.0%
4,Edge,0.9%,-2.1%,-0.5%,-3.6%,-5.2%,-9.0%
5,Outcome,2R,-8R,-2R,-20R,-32R,-110R


,SL bigger than 15 pips,1:1 RRR,1:2 RRR,1:3 RRR,1:4 RRR,1:5 RRR,1:10 RRR
0,Total trades,36,36,36,36,36,36
1,Wins,19,11,9,7,4,0
2,Losses,17,25,27,29,32,36
3,Win Rate,52.8%,30.6%,25.0%,19.4%,11.1%,0.0%
4,Edge,2.8%,-2.4%,0.0%,-0.6%,-5.9%,-9.0%
5,Outcome,2R,-3R,0R,-1R,-12R,-36R


In [86]:
# Let's run the updated cell to see the new output
exec(open('/Users/remigijus/Work/tweak-things/forex-backtester/forex_analysis.ipynb').read())